# Protein ↔ Protein Relation-Wise Merge

Merges Protein–Protein triples from Monarch, CKG (×3), CrossBAR, TARKG, DtiNet, and STITCH;
fills missing head/tail names from UniProt; deduplicates by `(head, relation, tail)`;
and saves the result.

## 0. Configuration

In [29]:
import pandas as pd
import numpy as np
import re
import os

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_PROTEIN/ALL_PROTEIN_PROTEIN.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Protein Lookup Dictionaries — UniProt

In [2]:
# RecName dict (AC → RecName)
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))

# Parsed TrEMBL dict (AC → NAME) for head/tail name fallback
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot

print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} | TrEMBL dict: {len(uniprot_trembl_AC_Name_dict):,}")

UniProt RecName dict: 794,151 | TrEMBL dict: 252,635,149


## 2. Load KG Sources

### 2a. Monarch

Head/tail and their UniProt columns are swapped in the raw file — corrected at load.

In [3]:
Monarch_Protein_Protein = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_Protein_Protein_Monarch.csv')
Monarch_Protein_Protein.columns = Monarch_Protein_Protein.columns.str.lower()
# Swap head/tail with their UniProt-AC counterparts so canonical AC is in head/tail
Monarch_Protein_Protein[['head', 'protein_uniprot_head']] = Monarch_Protein_Protein[['protein_uniprot_head', 'head']]
Monarch_Protein_Protein[['tail', 'protein_uniprot_tail']] = Monarch_Protein_Protein[['protein_uniprot_tail', 'tail']]
Monarch_Protein_Protein.rename(columns={'tail_name': 'tail_detail_name', 'head_name': 'head_detail_name', 'kgsource': 'kg_source'}, inplace=True)
print(f"Monarch:  {len(Monarch_Protein_Protein):,} rows")

Monarch_Protein_Protein['kg_type'] = 'Generalised'
Monarch_Protein_Protein['kg_source'] = 'Monarch_KG'
Monarch_Protein_Protein['species'] = 'Homo species'
Monarch_Protein_Protein.head(2)

Monarch:  18 rows


,head,tail,relation_type,relation_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,protein_uniprot_tail,protein_uniprot_head,kg_type,species
0,Q16950,Q25414,subclass_of,infores:pr,Monarch_KG,5-hydroxytryptamine receptor 1,Protein,PR,NaN,NaN,...,NaN,NaN,Protein,Protein,Protein_Protein,NaN,PR:000044625,PR:000001098,Generalised,Homo species
1,Q16951,Q25414,subclass_of,infores:pr,Monarch_KG,5-hydroxytryptamine receptor 2,Protein,PR,NaN,NaN,...,NaN,NaN,Protein,Protein,Protein_Protein,NaN,PR:000044625,PR:000001099,Generalised,Homo species


### 2b. CKG (×3)

In [4]:
def load_ckg_pp(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower()
    df.rename(columns={'tail_alt_name': 'tail_detail_name', 'head_alt_name': 'head_detail_name', 'kgsource': 'kg_source'}, inplace=True)
    return df

CKG_Protein_Protein1 = load_ckg_pp2(PROC_DIR + 'CKG/File_29_Protein_Protein_CKG.csv')
CKG_Protein_Protein2 = load_ckg_pp(PROC_DIR + 'CKG/File_33_Protein_Protein_CKG.csv')
CKG_Protein_Protein3 = load_ckg_pp(PROC_DIR + 'CKG/File_40_Protein_Protein_CKG.csv')

print(f"CKG 1: {len(CKG_Protein_Protein1):,} | CKG 2: {len(CKG_Protein_Protein2):,} | CKG 3: {len(CKG_Protein_Protein3):,}")

CKG 1: 516,029 | CKG 2: 670,002 | CKG 3: 52,090,442


In [8]:
CKG_Protein_Protein1['kg_type'] = 'Generalised'
CKG_Protein_Protein1['kg_source'] = 'CKG'
CKG_Protein_Protein1['species'] = 'Homo species'
CKG_Protein_Protein1


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,Q9H0P0-4,IS_ISOFORM,Q9H0P0,Protein,Protein,UniProt,CKG,NaN,Cytosolic 5'-nucleotidase 3A {ECO:0000305|PubM...,Uniprot_ID,Uniprot_ID,Generalised,Homo species
1,P49902-2,IS_ISOFORM,P49902,Protein,Protein,UniProt,CKG,NaN,Cytosolic purine 5'-nucleotidase {ECO:0000305|...,Uniprot_ID,Uniprot_ID,Generalised,Homo species
2,P17174-1,IS_ISOFORM,P17174,Protein,Protein,UniProt,CKG,NaN,"Aspartate aminotransferase, cytoplasmic {ECO:0...",Uniprot_ID,Uniprot_ID,Generalised,Homo species
3,P05067-10,IS_ISOFORM,P05067,Protein,Protein,UniProt,CKG,NaN,C31,Uniprot_ID,Uniprot_ID,Generalised,Homo species
4,Q5QJU3-3,IS_ISOFORM,Q5QJU3-3,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
516024,Q03936-1,IS_ISOFORM,Q03936-2,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID,Generalised,Homo species
516025,Q16670-1,IS_ISOFORM,Q16670-1,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID,Generalised,Homo species
516026,P17040-2,IS_ISOFORM,P17040-1,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID,Generalised,Homo species
516027,Q9UDY2-5,IS_ISOFORM,Q9UDY2-4,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID,Generalised,Homo species


In [9]:
CKG_Protein_Protein2['kg_type'] = 'Generalised'
CKG_Protein_Protein2['kg_source'] = 'CKG'
CKG_Protein_Protein2['species'] = 'Homo species'
CKG_Protein_Protein2

,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,P13498,CURATED_INTERACTS_WITH,Q8NFA2,Protein,Protein,MINT,CKG,Cytochrome b-245 light chain,NADPH oxidase organizer 1,Uniprot_ID,Uniprot_ID,Generalised,Homo species
1,Q8TAU3,CURATED_INTERACTS_WITH,Q5JXC2,Protein,Protein,IntAct,CKG,Zinc finger protein 417,Migration and invasion-inhibitory protein,Uniprot_ID,Uniprot_ID,Generalised,Homo species
2,Q9NWW5,CURATED_INTERACTS_WITH,P39656,Protein,Protein,IntAct,CKG,Ceroid-lipofuscinosis neuronal protein 6,Dolichyl-diphosphooligosaccharide--protein gly...,Uniprot_ID,Uniprot_ID,Generalised,Homo species
3,P14678,CURATED_INTERACTS_WITH,Q8WUD4,Protein,Protein,UniProt,CKG,Small nuclear ribonucleoprotein-associated pro...,Coiled-coil domain-containing protein 12,Uniprot_ID,Uniprot_ID,Generalised,Homo species
4,P67870,CURATED_INTERACTS_WITH,Q8WXI9,Protein,Protein,UniProt,CKG,Casein kinase II subunit beta,Transcriptional repressor p66-beta,Uniprot_ID,Uniprot_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
669997,P17948,CURATED_INTERACTS_WITH,Q13610,Protein,Protein,IntAct,CKG,Vascular endothelial growth factor receptor 1,Periodic tryptophan protein 1 homolog {ECO:000...,Uniprot_ID,Uniprot_ID,Generalised,Homo species
669998,O60895,CURATED_INTERACTS_WITH,Q12772,Protein,Protein,IntAct,CKG,Receptor activity-modifying protein 2,Processed sterol regulatory element-binding pr...,Uniprot_ID,Uniprot_ID,Generalised,Homo species
669999,Q96EK5,CURATED_INTERACTS_WITH,O43896,Protein,Protein,IntAct,CKG,KIF-binding protein,Kinesin-like protein KIF1C,Uniprot_ID,Uniprot_ID,Generalised,Homo species
670000,P41218,CURATED_INTERACTS_WITH,Q96GQ7,Protein,Protein,IntAct,CKG,Myeloid cell nuclear differentiation antigen,Probable ATP-dependent RNA helicase DDX27,Uniprot_ID,Uniprot_ID,Generalised,Homo species


In [10]:
CKG_Protein_Protein3['kg_type'] = 'Generalised'
CKG_Protein_Protein3['kg_source'] = 'CKG'
CKG_Protein_Protein3['species'] = 'Homo species'
CKG_Protein_Protein3

,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,evidence,score,kg_type,species
0,P84085,COMPILED_TARGETS,Q15191,Protein,Protein,STRING,CKG,ADP-ribosylation factor 5,Cyclin-dependent kinase inhibitor 2A {ECO:0000...,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.606,Generalised,Homo species
1,P84085,COMPILED_TARGETS,P42771,Protein,Protein,STRING,CKG,ADP-ribosylation factor 5,Cyclin-dependent kinase inhibitor 2A {ECO:0000...,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.606,Generalised,Homo species
2,P84085,COMPILED_TARGETS,Q9NP05,Protein,Protein,STRING,CKG,ADP-ribosylation factor 5,Cyclin-dependent kinase inhibitor 2A {ECO:0000...,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.606,Generalised,Homo species
3,P84085,COMPILED_TARGETS,O95440,Protein,Protein,STRING,CKG,ADP-ribosylation factor 5,Cyclin-dependent kinase inhibitor 2A {ECO:0000...,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.606,Generalised,Homo species
4,P84085,COMPILED_TARGETS,A5X2G7,Protein,Protein,STRING,CKG,ADP-ribosylation factor 5,Cyclin-dependent kinase inhibitor 2A {ECO:0000...,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.606,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52090437,Q96R34,COMPILED_TARGETS,Q9NYF2,Protein,Protein,STRING,CKG,Olfactory receptor 6Q1,Receptor expression-enhancing protein 2,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.900,Generalised,Homo species
52090438,Q96R34,COMPILED_TARGETS,Q9BRK0,Protein,Protein,STRING,CKG,Olfactory receptor 6Q1,Receptor expression-enhancing protein 2,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.900,Generalised,Homo species
52090439,Q6IFH1,COMPILED_TARGETS,Q53EM8,Protein,Protein,STRING,CKG,Olfactory receptor 6Q1,Receptor expression-enhancing protein 2,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.900,Generalised,Homo species
52090440,Q6IFH1,COMPILED_TARGETS,Q9NYF2,Protein,Protein,STRING,CKG,Olfactory receptor 6Q1,Receptor expression-enhancing protein 2,UniprotID,UniprotID,"Neighborhood in the Genome,Gene fusions,Co-ocu...",0.900,Generalised,Homo species


### 2c. CrossBAR

In [11]:
CrossBAR_Protein_Protein = pd.read_csv(PROC_DIR + 'crossbar/Protein_Protein_Crossbar.csv')
CrossBAR_Protein_Protein.columns = CrossBAR_Protein_Protein.columns.str.lower()
print(f"CrossBAR: {len(CrossBAR_Protein_Protein):,} rows")
CrossBAR_Protein_Protein['kg_type'] = 'Generalised'
CrossBAR_Protein_Protein['kg_source'] = 'crossbar'
CrossBAR_Protein_Protein['species'] = 'Homo species'
CrossBAR_Protein_Protein.head(2)


CrossBAR: 292,762 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,A0AV96,Protein_Protein,Q13315,Protein,Protein,crossbar,RNA-binding protein 47 {ECO:0000305|PubMed:249...,Serine-protein kinase ATM,Uniprot_ID,Uniprot_ID,Generalised,Homo species
1,A0AV96,Protein_Protein,P57721,Protein,Protein,crossbar,RNA-binding protein 47 {ECO:0000305|PubMed:249...,Poly(rC)-binding protein 3,Uniprot_ID,Uniprot_ID,Generalised,Homo species


### 2d. TARKG

In [13]:
TARKG_Protein_Protein = pd.read_csv(PROC_DIR + 'TARKG/Protein_Protein_TARKG.csv')
TARKG_Protein_Protein.columns = TARKG_Protein_Protein.columns.str.lower()
print(f"TARKG:    {len(TARKG_Protein_Protein):,} rows")
TARKG_Protein_Protein['kg_type'] = 'Generalised'
TARKG_Protein_Protein['kg_source'] = 'TARKG'
TARKG_Protein_Protein['species'] = 'Homo species'
TARKG_Protein_Protein.head(2)

TARKG:    1,271,469 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,Q9Y4X5,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,E3 ubiquitin-protein ligase ARIH1,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522140,addKG,0,Generalised,Homo species
1,Q99728,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,BRCA1-associated RING domain protein 1,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522143,addKG,0,Generalised,Homo species


### 2e. DtiNet

In [16]:
DtiNet_Protein_Protein = pd.read_csv(PROC_DIR + 'DTINet/Protein_Protein_DTINet.csv')
DtiNet_Protein_Protein.columns = DtiNet_Protein_Protein.columns.str.lower()
print(f"DtiNet:   {len(DtiNet_Protein_Protein):,} rows")
DtiNet_Protein_Protein['kg_type'] = 'Generalised'
# DtiNet_Protein_Protein['kg_source'] = 'DTINet'
DtiNet_Protein_Protein['species'] = 'Homo species'
DtiNet_Protein_Protein.head(2)

DtiNet:   7,232 rows


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source,kg_type,species
0,Q9UI32,Protein_Protein,O14936,Protein,Protein,DTINet,Q9UI32,"Glutaminase liver isoform, mitochondrial",Peripheral plasma membrane protein CASK {ECO:0...,Uniprot_ID,Uniprot_ID,DTINet,Generalised,Homo species
1,Q9UI32,Protein_Protein,P78352,Protein,Protein,DTINet,Q9UI32,"Glutaminase liver isoform, mitochondrial",Disks large homolog 4,Uniprot_ID,Uniprot_ID,DTINet,Generalised,Homo species


### 2f. STITCH / STRING

In [19]:
STITCH_Protein_Protein = pd.read_csv(PROC_DIR + 'string/hsap/Protein_Protein_STRING.csv')
STITCH_Protein_Protein.columns = STITCH_Protein_Protein.columns.str.lower()
print(f"STITCH:   {len(STITCH_Protein_Protein):,} rows")
STITCH_Protein_Protein['kg_type'] = 'Generalised'
STITCH_Protein_Protein['kg_source'] = 'string'
STITCH_Protein_Protein['species'] = 'Homo species'
STITCH_Protein_Protein.head(2)

STITCH:   5,101,030 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,P84085,Protein_Protein,Q86X27,Protein,Protein,string,ADP-ribosylation factor 5,Ras-specific guanine nucleotide-releasing fact...,Uniprot_ID,Uniprot_ID,Generalised,Homo species
1,P84085,Protein_Protein,Q9C0D6,Protein,Protein,string,ADP-ribosylation factor 5,FH2 domain-containing protein 1,Uniprot_ID,Uniprot_ID,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [22]:
SOURCE_DFS = [
    ('Monarch_Protein_Protein',  Monarch_Protein_Protein),
    ('CKG_Protein_Protein1',     CKG_Protein_Protein1),
    ('CKG_Protein_Protein2',     CKG_Protein_Protein2),
    ('CKG_Protein_Protein3',     CKG_Protein_Protein3),
    ('CrossBAR_Protein_Protein', CrossBAR_Protein_Protein),
    ('TARKG_Protein_Protein',    TARKG_Protein_Protein),
    ('DtiNet_Protein_Protein',   DtiNet_Protein_Protein),
    ('STITCH_Protein_Protein',   STITCH_Protein_Protein),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Protein_Protein] ✓ no duplicates
[CKG_Protein_Protein1] ✓ no duplicates
[CKG_Protein_Protein2] ✓ no duplicates
[CKG_Protein_Protein3] ✓ no duplicates
[CrossBAR_Protein_Protein] ✓ no duplicates
[TARKG_Protein_Protein] ✓ no duplicates
[DtiNet_Protein_Protein] ✓ no duplicates
[STITCH_Protein_Protein] ✓ no duplicates


## 4. Align Schemas and Concatenate

In [23]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 59,948,984 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q16950,Protein_Protein,Q25414,Protein,subclass_of,Protein,Monarch_KG,PR,PR,5-hydroxytryptamine receptor 1,5-hydroxytryptamine receptor,Generalised,Homo species
1,Q16951,Protein_Protein,Q25414,Protein,subclass_of,Protein,Monarch_KG,PR,PR,5-hydroxytryptamine receptor 2,5-hydroxytryptamine receptor,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [24]:
final_df['head_type']  = 'Protein'
final_df['tail_type']  = 'Protein'
final_df['relation']   = 'Protein_Protein'
final_df['head_id_is'] = 'Uniprot_ID'
final_df['tail_id_is'] = 'Uniprot_ID'

print("Unique relation:",  set(final_df['relation']))
print("Unique kg_source:", set(final_df['kg_source']))
print("Shape:", final_df.shape)

Unique relation: {'Protein_Protein'}
Unique kg_source: {'TARKG', 'CKG', 'crossbar', 'DTINet', 'Monarch_KG', 'string'}
Shape: (59948984, 13)


## 6. Deduplicate

In [25]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 58,354,854 rows


## 7. Fill Missing Head/Tail Names from UniProt



In [26]:
# Normalise fake NaNs to real np.nan, then fill from UniProt AC→Name dict
for node in ['head', 'tail']:
    col = f'{node}_detail_name'
    final_df_dedup[col] = final_df_dedup[col].replace(['NAN', 'NaN', 'None'], np.nan)
    final_df_dedup[col] = final_df_dedup[col].fillna(final_df_dedup[node].map(uniprot_trembl_AC_Name_dict))

# Drop rows still missing either name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[
    ~final_df_dedup['head_detail_name'].isna() &
    ~final_df_dedup['tail_detail_name'].isna()
].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing names → {len(final_df_dedup):,} remaining")

Dropped 329,125 rows with missing names → 58,025,729 remaining


## 8. Add Schema Columns and Enforce Column Order

In [27]:

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (58025729, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A024R072,Protein_Protein,A0A0A0MRF8,Protein,None,Protein,CKG,Uniprot_ID,Uniprot_ID,Cyclin-L2,Cytochrome c oxidase subunit 1 {ECO:0000256|AR...,Generalised,Homo species
1,A0A024R072,Protein_Protein,A2A389,Protein,None,Protein,CKG,Uniprot_ID,Uniprot_ID,Cyclin-L2,Cyclin-dependent kinase 20,Generalised,Homo species
2,A0A024R072,Protein_Protein,A2A390,Protein,None,Protein,CKG,Uniprot_ID,Uniprot_ID,Cyclin-L2,Cyclin-dependent kinase 20,Generalised,Homo species
3,A0A024R072,Protein_Protein,A4D1E6,Protein,None,Protein,CKG,Uniprot_ID,Uniprot_ID,Cyclin-L2,Cyclin-dependent kinase 14,Generalised,Homo species
4,A0A024R072,Protein_Protein,A4D1G0,Protein,None,Protein,CKG,Uniprot_ID,Uniprot_ID,Cyclin-L2,Cyclin-dependent kinase 6,Generalised,Homo species


## 9. QC — NaN Check

In [28]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,56765979,0,56765979
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [30]:
#

In [31]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 58,025,729 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_PROTEIN/ALL_PROTEIN_PROTEIN.parquet


In [32]:
#